# Building a Basic Brick

In this tutorial, we'll go through a simple example of how to build a basic AND brick. We'll start with some basic imports. We almost always use NetworkX and Numpy.

In [ ]:
# General imports
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

import fugu
from fugu import Scaffold, Brick
from fugu.bricks import Vector_Input
from fugu.backends import snn_Backend
from fugu.scaffold import ChannelSpec, PortSpec, ChannelData, PortData, PortUtil

All developed bricks should inherit from the `Brick` class.  Bricks that are listed as input bricks should instead inherit from `InputBrick` (which is beyond the scope of this tutorial).

The construction of bricks is relatively systematic:
- The base Brick class provides the structure for the scaffold to build a neural graph from it (and add it to the global graph)
- Subclasses of Brick should additionally provide the actual code that will generate the nodes and edges in the graph (through NetworkX)
  - This takes place within the `__init__` and  `build` methods
  - Additionally, input/output ports are defined as `classmethods`

Let's look at a simplified definition of the base `Brick` class:

```python 
class Brick(ABC):
    def __init__(self, name="Brick"):
        self.name = name
        self.is_built = False
        self.supported_codings = 
        
    @classmethod
    def input_ports(cls) -> dict[str, PortSpec]:
        return {}

    @classmethod
    def output_ports(cls) -> dict[str, PortSpec]:
        return {}

    @abstractmethod
    def build2(self, graph, inputs: dict[str, PortData] = {}):
        pass
```

The first line `class Brick(ABC)` defines the abstract class of `Brick`.  Brick objects inherit from `ABC` which just means that `Brick` is an abstract class that cannot be instantiated on its own; only subclasses may be instantiated.

The ``__init__`` method contains standard instantiation code.  All bricks are expected to have a member property `self.name` that is unique to the brick.  The uniqueness needs to be determined by the scaffold, not by the brick.

The property `self.is_built` is a boolean that is `True` if the brick has been built (added to the graph). This starts off as `False` during initialization, and is changed to `True` after the the brick finishes its `build` method.

The property `self.supported_codings` is a list of input codings (strings) that the brick supports.  Since you have the full use of python when you are defining your brick, you can support multiple coding types completely transparent to the user.

The class methods `input_ports()` and `output_ports()` describe the expected inputs that a brick receives and the outputs that brick provides/generates, respectively. This is how bricks pass neurons externally between other bricks for compositionality and coordination.

The method `build2` will be called by the scaffold when the graph is to be built.  Arguments are:
- graph: The neuron graph object that we are building onto (this currently uses NetworkX).
- inputs: A *dict* of input ports. Each *PortData* object describes the structure of one port, along with lists of concrete neuron IDs.

Each brick is responsible for throwing the appropriate errors/warnings if the inputs are not compatible with the brick.

In [ ]:
# A full list of coding types
fugu.input_coding_types

## Example AND Brick

In [ ]:
class AND(Brick):
    # initialization method: name is required, additional arguments are subclass specific
    def __init__(self, name="AND", other_params=None):
        super().__init__(name)              # initialize the base class (must be called)
        self.other_params = other_params    # these may be used during building

    # These methods describes the input ports that any AND brick expects, and
    # the output ports that any AND brick provides. Here, this includes the
    # actual (spike) data, and a control signal for 'completed' processing
    @classmethod
    def input_ports(cls) -> dict[str, PortSpec]:
        port = PortSpec(name='input', minimum=2, maximum=2)     # The number of ports
        port.channels['data']     = ChannelSpec(name='data')    # is subclass specific
        port.channels['complete'] = ChannelSpec(name='complete', shape=(1,))
        return {port.name: port}

    @classmethod
    def output_ports(cls) -> dict[str, PortSpec]:
        port = PortSpec(name='output')           # You can also include the encoding type
        port.channels['data']     = ChannelSpec(name='data', coding=['Raster'])
        port.channels['complete'] = ChannelSpec(name='complete', shape=(1,))
        return {port.name: port}

    # This method does the actual network construction. It reads the input ports,
    # wires up neurons to the graph, and passes neurons via the output port
    def build2(self, graph, inputs: dict[str, PortData] = {}):
        # For the AND operation, we expect exactly two inputs
        if len(inputs) != 2:
            raise ValueError('Only two inputs supported.')

        # Unpack the input ports into convenience variables
        input1, input2 = PortUtil.get_autoports(inputs, 'input', 2)
        input1_neurons = input1.channels['data'].neurons
        input2_neurons = input2.channels['data'].neurons

        # Create our output port (and convenience variables)
        outputs = PortUtil.make_ports_from_specs(AND.output_ports())
        output = outputs['output'] # Unpack the only actual output port
        output_neurons = output.channels['data'].neurons
        # set the coding spec for the output port (here to the same as the inputs)
        output.channels['data'].spec.coding = input1.channels['data'].spec.coding

        # Hook up the control signals (when a brick completes its processing)
        # Here, we just forward the incoming signal with one cycle of delay

        # We generate node/neuron names using the helper function
        # generate_neuron_name to get a unique name for NetworkX
        complete_node_name = self.generate_neuron_name('complete')
        output.channels['complete'].neurons = [complete_node_name]

        # Work with the NetworkX graph directly by adding neurons
        graph.add_node(complete_node_name,   # neuron name
                       index=-1,             # neuron parameters
                       threshold=0.0,
                       decay=0.0, p=1.0,     # the index for control signals
                       potential=0.0)        # are negative by convention

        # Connect the neurons by referencing their names
        graph.add_edge(input1_neurons[0],    # source neuron
                       complete_node_name,   # target neuron
                       weight=1.0, delay=1)  # synapse parameters

        # Build the computational graph, this is the "main" part of the brick
        # Here, the idea is to iterate over our two input data channels in parallel,
        # connecting each pair of inputs via a neuron that does the AND operation
        for i in range(min(len(input1_neurons), len(input2_neurons))):
            operand1 = input1_neurons[i]
            operand2 = input2_neurons[i]
            
            # Generate a name for the AND neuron (based on their input names)
            and_node_name = self.generate_neuron_name(f"{operand1}_{operand2}")
            # This will also be an output neuron (so we add it to the output list)
            output_neurons.append(and_node_name)
            
            # Create the element-wise AND neuron
            graph.add_node(and_node_name, index=i,
                           threshold=1.0, decay=1.0, p=1.0, potential=0.0)
            # Make a synapse from the first input neuron.
            graph.add_edge(operand1, and_node_name,
                           weight=0.75, delay=1.0)
            # Make a synapse from the second input neuron.
            graph.add_edge(operand2, and_node_name,
                           weight=0.75, delay=1.0)

        # At this point, the network has been built
        # We return the output ports
        self.is_built = True
        return outputs

### Port information
Brick classes provide a means for _reflection_ through their class methods. We can ask them for their port specification as a data structure by calling input_ports() and output_ports(). Or we can ask for a human-readable description.

In [ ]:
# Show port information
AND.show_ports()

## Simple scaffold

Knowing the ports, we can assemble a simple scaffold to connect bricks together, construct our network procedurally, and simulate it.

In [ ]:
# Fugu scaffold object
scaffold = Scaffold()

# Input bricks
I1 = scaffold.add_brick(Vector_Input(np.array([1,0,1,0]), coding='Raster', name='input1'))
I2 = scaffold.add_brick(Vector_Input(np.array([1,1,0,0]), coding='Raster', name='input2'))

# The AND brick we developed
A = scaffold.add_brick(AND(), output=True)

# Connect the bricks
scaffold.connect(I1, A)  # Binds I1's default output port to A's first default input port
scaffold.connect(I2, A)  # Binds I2's default output port to A's second default input port
                         # These can be manually specified with from/to port names
# Build the bricks
scaffold.lay_bricks()
scaffold.summary(verbose=1)

In [ ]:
# We can now simulate our network
backend = snn_Backend()
backend_args = {}
backend_args['record'] = 'all'
backend.compile(scaffold, backend_args)
result = backend.run(10)
print(result)